# Image Vectorizer — Interactive UI (Optional)

> **Note:** The primary, recommended way to use this tool is the command-line script — see [run.py](../run.py) and the main [README](../README.md). This notebook is an optional, secondary interface.
>
> **Known limitation:** In VS Code's built-in notebook UI, widgets (sliders, upload button) require fetching JavaScript from `unpkg.com`. On networks that block this CDN, widgets fail to render and the cell may appear to crash. If that happens, either run this notebook in **JupyterLab** instead (`jupyter lab notebooks/vectorizer_ui.ipynb`, which serves widget assets locally), or use `run.py` from the command line.

Convert technical diagram images (PNG / JPG) into clean SVG vector graphics.

**Steps:**
1. Run all cells once — **Kernel → Restart & Run All**
2. Upload images with the **Upload Images** button
3. Adjust parameters if needed *(defaults work well for clean CAD exports)*
4. Click **▶ Process Images** and review the preview
5. Click **⬇ Download ZIP** to save all results

In [ ]:
# Run this cell once to install all required packages into your current kernel.
# Safe to re-run — skips packages that are already installed.
%pip install opencv-python numpy Pillow ipywidgets matplotlib vtracer --quiet

In [ ]:
import sys
from pathlib import Path

# Resolve project root so 'src.image_vectorizer' is importable
# whether Jupyter was launched from the repo root or from notebooks/
_here = Path.cwd()
_root = _here if (_here / 'src').exists() else _here.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import tempfile
import zipfile

import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, FileLink
from PIL import Image

from src.image_vectorizer import process_image, default_params

_output_dir = Path(tempfile.mkdtemp(prefix='vectorizer_'))
_results: list = []

print(f'Ready. Outputs will be written to: {_output_dir}')


In [ ]:
# ── Tab 1: Threshold & Contrast ─────────────────────────────────────────
w_invert     = widgets.Checkbox(value=True,  description='Invert (dark lines on white bg)', indent=False)
w_use_otsu   = widgets.Checkbox(value=True,  description='Auto threshold (Otsu — recommended)', indent=False)
w_threshold  = widgets.IntSlider(
    value=240, min=50, max=254, description='Manual threshold',
    style={'description_width': '160px'}, layout=widgets.Layout(width='420px'))
w_use_adap   = widgets.Checkbox(value=False, description='Adaptive threshold (for uneven scans)', indent=False)
w_adap_block = widgets.IntSlider(
    value=25, min=3, max=101, step=2, description='Block size',
    style={'description_width': '160px'}, layout=widgets.Layout(width='420px'))
w_adap_c     = widgets.IntSlider(
    value=10, min=0, max=30, description='C offset',
    style={'description_width': '160px'}, layout=widgets.Layout(width='420px'))
w_clahe      = widgets.Checkbox(value=False, description='CLAHE (contrast boost for faded scans)', indent=False)
w_median     = widgets.IntSlider(
    value=0, min=0, max=9, step=2, description='Median blur',
    style={'description_width': '160px'}, layout=widgets.Layout(width='420px'))

tab_threshold = widgets.VBox([
    w_invert, w_use_otsu, w_threshold,
    w_use_adap, w_adap_block, w_adap_c,
    w_clahe, w_median,
])

# ── Tab 2: Cleanup ────────────────────────────────────────────────────────
w_noise_k   = widgets.IntSlider(
    value=0, min=0, max=10, description='Close gaps (CLOSE)',
    style={'description_width': '160px'}, layout=widgets.Layout(width='420px'))
w_speckle_k = widgets.IntSlider(
    value=0, min=0, max=10, description='Remove speckles (OPEN)',
    style={'description_width': '160px'}, layout=widgets.Layout(width='420px'))
w_upscale   = widgets.Dropdown(
    options=[('1x (off)', 1), ('2x (recommended)', 2), ('3x', 3)],
    value=2, description='Upscale',
    style={'description_width': '160px'})
w_skeleton  = widgets.Checkbox(value=False, description='Skeletonize (thin strokes to 1 px)', indent=False)
w_debug     = widgets.Checkbox(value=False, description='Save debug contact sheet (_debug.png)', indent=False)

tab_cleanup = widgets.VBox([w_noise_k, w_speckle_k, w_upscale, w_skeleton, w_debug])

# ── Tab 3: Vectorization ──────────────────────────────────────────────────
w_turdsize = widgets.IntSlider(
    value=2, min=0, max=20, description='Suppress small paths',
    style={'description_width': '160px'}, layout=widgets.Layout(width='420px'))
w_alphamax = widgets.FloatSlider(
    value=0.0, min=0.0, max=1.33, step=0.01, description='Corner smoothness',
    style={'description_width': '160px'}, layout=widgets.Layout(width='420px'))
w_opttol   = widgets.FloatSlider(
    value=0.1, min=0.05, max=1.0, step=0.05, description='Curve tolerance',
    style={'description_width': '160px'}, layout=widgets.Layout(width='420px'))

tab_vector = widgets.VBox([w_turdsize, w_alphamax, w_opttol])

# ── Tab 4: Text Detection ─────────────────────────────────────────────────
w_detect = widgets.Checkbox(
    value=True, description='Detect text label regions (L1, L2…)', indent=False)

tab_text = widgets.VBox([w_detect])

# ── Tab container ─────────────────────────────────────────────────────────
param_tabs = widgets.Tab(children=[tab_threshold, tab_cleanup, tab_vector, tab_text])
for _i, _t in enumerate(['Threshold & Contrast', 'Cleanup', 'Vectorization', 'Text Detection']):
    param_tabs.set_title(_i, _t)


In [ ]:
w_upload   = widgets.FileUpload(
    accept='image/*', multiple=True,
    description='Upload Images',
    layout=widgets.Layout(width='200px'))
w_run_btn  = widgets.Button(
    description='▶  Process Images', button_style='primary',
    layout=widgets.Layout(width='180px'))
w_dl_btn   = widgets.Button(
    description='⬇  Download ZIP', button_style='success',
    disabled=True, layout=widgets.Layout(width='180px'))
w_progress = widgets.IntProgress(
    value=0, min=0, max=1, description='Progress:',
    layout=widgets.Layout(width='460px'))
w_status   = widgets.HTML(value='<span style="color:#666">No files processed yet.</span>')
w_output   = widgets.Output()


In [ ]:
def _collect_params() -> dict:
    p = default_params()
    p.update({
        'invert':                 w_invert.value,
        'use_otsu':               w_use_otsu.value,
        'threshold':              w_threshold.value,
        'use_adaptive_threshold': w_use_adap.value,
        'adaptive_block_size':    w_adap_block.value,
        'adaptive_c':             w_adap_c.value,
        'use_clahe':              w_clahe.value,
        'median_blur':            w_median.value,
        'noise_kernel':           w_noise_k.value,
        'speckle_kernel':         w_speckle_k.value,
        'upscale_factor':         w_upscale.value,
        'skeletonize':            w_skeleton.value,
        'detect_text':            w_detect.value,
        'potrace_turdsize':       w_turdsize.value,
        'potrace_alphamax':       w_alphamax.value,
        'potrace_opttolerance':   w_opttol.value,
    })
    return p


def _parse_uploads(widget):
    val = widget.value
    if isinstance(val, dict):                          # ipywidgets v7
        return [(name, info['content']) for name, info in val.items()]
    return [(f['name'], f['content']) for f in val]   # ipywidgets v8


def _preview(result: dict) -> None:
    orig  = Image.open(result['input']).convert('RGB')
    clean = Image.open(result['png']).convert('RGBA')
    bg    = Image.new('RGB', clean.size, (255, 255, 255))
    bg.paste(clean, mask=clean.split()[3])

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(orig);  axes[0].set_title(f"Original   {Path(result['input']).name}",  fontsize=9)
    axes[1].imshow(bg);    axes[1].set_title(f"Cleaned    {Path(result['png']).name}",     fontsize=9)
    for ax in axes: ax.axis('off')
    plt.tight_layout()
    plt.show()


def on_run(btn):
    global _results
    _results = []
    files = _parse_uploads(w_upload)

    if not files:
        w_status.value = '<span style="color:orange">⚠ Upload at least one image first.</span>'
        return

    params           = _collect_params()
    save_debug       = w_debug.value
    w_progress.max   = len(files)
    w_progress.value = 0
    w_dl_btn.disabled = True
    w_output.clear_output()

    with w_output:
        for name, content in files:
            w_status.value = f'Processing <b>{name}</b>…'
            tmp = _output_dir / name
            tmp.write_bytes(content)
            try:
                result = process_image(tmp, _output_dir, params=params, save_debug=save_debug)
                _results.append(result)
                _preview(result)
                n_text = len(result.get('text_boxes', []))
                print(f"  [{result['engine']}]  {name}  —  {n_text} text region(s)")
            except Exception as exc:
                print(f'  ERROR — {name}: {exc}')
            w_progress.value += 1

    count = len(_results)
    w_status.value = (
        f'<span style="color:green">✓ {count} file(s) processed.</span>'
        if count else
        '<span style="color:red">All files failed — check the output above.</span>'
    )
    w_dl_btn.disabled = (count == 0)


def on_download(btn):
    zip_path = _output_dir / 'vectorizer_results.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for result in _results:
            stem = Path(result['input']).stem
            for p in [
                _output_dir / f'{stem}_clean.png',
                _output_dir / f'{stem}.svg',
                _output_dir / f'{stem}_text.json',
                _output_dir / f'{stem}_debug.png',
            ]:
                if p.exists():
                    zf.write(p, p.name)
    with w_output:
        w_output.clear_output()
        try:
            display(FileLink(str(zip_path), result_html_prefix='⬇ Download: '))
        except Exception:
            print(f'ZIP saved to:\n{zip_path}')


w_run_btn.on_click(on_run)
w_dl_btn.on_click(on_download)


In [ ]:
display(widgets.VBox([
    widgets.HTML("<h3 style='margin: 0 0 6px'>Step 1 — Upload images</h3>"),
    w_upload,
    widgets.HTML("<h3 style='margin: 16px 0 6px'>Step 2 — Set parameters</h3>"),
    param_tabs,
    widgets.HTML("<h3 style='margin: 16px 0 6px'>Step 3 — Process &amp; download</h3>"),
    widgets.HBox([w_run_btn, w_dl_btn], layout=widgets.Layout(gap='12px')),
    w_progress,
    w_status,
    w_output,
], layout=widgets.Layout(padding='16px', max_width='720px')))
